# Testing and debugging the dataset creation

In [14]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
import json
from datetime import datetime
import re
from urllib.parse import urlsplit
import time
import sys
sys.path.append("..")
import importlib
import scripts.ingest

importlib.reload(scripts.ingest)

from scripts.ingest import (
    processed_date,
    scrape_archive_metadata,
    selection_logic,
    extract_slug,
    scrape_or_load_edition,
    filter_and_clean_parts,
    url_headline_extract,
    process_edition,
    append_and_store_edition,
    append_and_store_quiz,
    check_existence_process_append_store
)

load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

client = Firecrawl(api_key=api_key)

data_path = "../data"

with open(f"{data_path}/archive.json", "r") as f:
    archive = json.load(f)

In [15]:
full_archive_metadata = scrape_archive_metadata(client)

In [147]:
# Constants and initial steps (just testing on one edition)

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday

data_path = "../data"

index = 18
edition_metadata = full_archive_metadata[index]

id = edition_metadata["_id"]
date = processed_date(edition_metadata)
print(date)
edition_metadata

2026-09-24


{'_id': 'nhT0Zu3AWutKiqVsav3v3q',
 'date': '2026-09-24T06:00:00-04:00',
 'previewText': 'The Chinese leader is in the US...',
 'slug': 'mr-xi-goes-to-washington',
 'subjectLine': 'Mr. Xi goes to Washington',
 'title': None,
 'websitePreviewText': None}

In [148]:
selection_logic(date)

True

In [149]:
slug = extract_slug(edition_metadata, date)
print(slug)

mr-xi-goes-to-washington


In [150]:
edition_metadata = full_archive_metadata[index]
edition_id = edition_metadata["_id"]
iso_date = date.isoformat()

scraped_edition = scrape_or_load_edition(client, slug, edition_id, date, archive)
scraped_edition

Edition for 2026-09-24 found


'[Skip to main content](https://www.morningbrew.com/issues/mr-xi-goes-to-washington#main-content)\n\n###### ☕Mr. Xi goes to Washington\n\nTo:Brew Readers\n\n* * *\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| |     |\n| --- |\n| The Chinese leader is in the US... |\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| ![Advertisement](https://ad.doubleclick.net/ddm/trackimp/N424004.3407870MORNINGBREW/B35376385.443322354;dc_trk_aid=638158829;dc_trk_cid=254254551;u=__AP1_np_ia_3913804507PA__;ord=6442997631;dc_lat=;dc_rdid=;tag_for_child_directed_treatment=;tfua=;gdpr=$%7BGDPR%7D;gdpr_consent=$%7BGDPR_CONSENT_755%7D;ltd=;dc_tdv=1?)![](https://pixel.adsafeprotected.com/rfw/st/2891071/93755519/skeleton.gif?gdpr=${GDPR}&gdpr_consent=${GDPR_CONSENT_278}&gdpr_pd=${GDPR_PD})![](https://tag.researchnow.com/t/beacon?pr=292479&adn=1&ca=35376385&si=8054551&pl=443322354&cr=254254551&did=AdvertiserId&ord=CacheBuster&gdpr=${GDPR}&gdpr_consent=${GDPR_CONSENT_110}&us_privacy=${US_PRIVA

In [151]:
split_quiz = scraped_edition.split("<br>")
split_quiz

['[Skip to main content](https://www.morningbrew.com/issues/mr-xi-goes-to-washington#main-content)\n\n###### ☕Mr. Xi goes to Washington\n\nTo:Brew Readers\n\n* * *\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| |     |\n| --- |\n| The Chinese leader is in the US... |\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| ![Advertisement](https://ad.doubleclick.net/ddm/trackimp/N424004.3407870MORNINGBREW/B35376385.443322354;dc_trk_aid=638158829;dc_trk_cid=254254551;u=__AP1_np_ia_3913804507PA__;ord=6442997631;dc_lat=;dc_rdid=;tag_for_child_directed_treatment=;tfua=;gdpr=$%7BGDPR%7D;gdpr_consent=$%7BGDPR_CONSENT_755%7D;ltd=;dc_tdv=1?)![](https://pixel.adsafeprotected.com/rfw/st/2891071/93755519/skeleton.gif?gdpr=${GDPR}&gdpr_consent=${GDPR_CONSENT_278}&gdpr_pd=${GDPR_PD})![](https://tag.researchnow.com/t/beacon?pr=292479&adn=1&ca=35376385&si=8054551&pl=443322354&cr=254254551&did=AdvertiserId&ord=CacheBuster&gdpr=${GDPR}&gdpr_consent=${GDPR_CONSENT_110}&us_privacy=${US_PRIV

In [136]:
q_search = re.compile(r'^[1234]\. (“)?.*\[([^\]]*)\](\(.*\))')
questions_and_answers = [item for item in split_quiz if q_search.search(item.lower())]
questions_and_answers

['1. "Man aims to see _Tenet_[120 times](https://www.upi.com/Odd_News/2020/09/01/Man-aims-to-see-Tenet-120-times-for-Guinness-record/2401598983881/?utm_source=morning_brew) for Guinness record"',
 '2. "Man facing charges for pulling $70k out of his children’s college fund to [cover up](https://media.giphy.com/media/OiiD2AVTbsOxa/giphy.gif?utm_source=morning_brew) risky Kodak trade"',
 '3. "An Ohio man built a [backyard squirrel bar](https://www.cnn.com/travel/article/squirrel-bar-tap-nuts-ohio-woodworking-trnd/index.html?utm_source=morning_brew) with seven varieties of nuts on tap"',
 '4. "Man tries to [rob pawnshop](https://www.boston25news.com/news/trending/man-tries-rob-pawnshop-with-sword-he-just-purchased-police-say/QLPPF3DJFJACFJL4I7EZ5FEGVY/?utm_source=morning_brew) with sword he just purchased, police say" | |\n| --- |\n\n| |     |\n| --- |\n| ### SHARE THE BREW |\n\n|     |\n| --- |\n| {if profile.vars.referral\\_count == 0}']

In [47]:
answer_search = re.compile(r'(\banswer)(s?)(\b)')
answer_hash_search = re.compile(r'(\s### answer)(s)?')
answer_hash_search2 = re.compile(r'(\s# \[answer\])(s)?')
answer_saturday_hash_search = re.compile(r'(\s### saturday headline)(s)?( answer)(s)?')
if (answer_search.search(questions_and_answers[3].lower()) 
or answer_hash_search.search(questions_and_answers[3].lower()) 
or answer_hash_search2.search(questions_and_answers[3].lower())):
    answer_rough = questions_and_answers[3].lower().split("answer")[1]
    answer_parts = answer_rough.split("|")
    answer_parts_stripped = [item.strip().strip("\n-") for item in answer_parts]
    answer_list = [item for item in answer_parts_stripped if item]
    answer = answer_list[0]
    print(answer)
else:
    print("next cell")

we made up the one about the demise of free bread.


In [40]:
answer = [item for item in split_quiz if (answer_search.search(item.lower()) and item.startswith("[morningbrew.com")) 
                                                or answer_hash_search.search(item.lower())
                                                or answer_hash_search2.search(item.lower())
                                                or answer_saturday_hash_search.search(item.lower())][0]

IndexError: list index out of range

In [56]:
questions_and_answers = filter_and_clean_parts(scraped_edition)
questions_and_answers

IndexError: list index out of range

In [57]:
url = re.compile(r'(\(.*?\))').search(questions_and_answers[0]).group().strip("()")
questions_and_answers[0].replace(url, "").replace("()", "").replace("[", "").replace("]", "").lstrip("1234. ")

'JD Vance calls ‘Emily in Paris’ the ‘Shakespeare of our time’'

In [ ]:
test = "Many letters included in this test string for testing purposes"
test[5][]

'l'

In [58]:
# quiz = {
#         "_id": edition_id,
#         "date": date.isoformat(),
#         "source_url": f"https://www.morningbrew.com/issues/{slug}",
#         "headlines": [],
#         "answer": questions_and_answers[4]
#     }

for index in range(4):
    if index < 4:
        headline, url = url_headline_extract(index, questions_and_answers)
        # quiz["headlines"].append({"text": headline, "url": url})
        print(headline, url)#quiz["headlines"].append({"text": headline, "url": url})

# quiz

JD Vance calls ‘Emily in Paris’ the ‘Shakespeare of our time’ https://www.dispatch.com/story/entertainment/2026/09/15/jd-vance-emily-in-paris-netflix-shakespeare/91778982007/
A Tennessee high school will pay for their homecoming dance with a hot dog eating contest https://giphy.com/gifs/ate-chestnut-10DTeHWvPQQIX6
Cash-strapped Japanese monks turn to investing as inflation worsens https://www.bloomberg.com/news/articles/2026-09-15/japan-s-cash-strapped-monks-pushed-to-invest-as-inflation-worsens
Netflix sets ‘ New Stanford Prison Experiment’ unscripted series for October release https://variety.com/2026/tv/news/netflix-stanford-prison-experiment-series-release-date-1236863011/


In [59]:
process_edition(client, index, full_archive_metadata, date, archive)

Edition for 2026-09-17 found


IndexError: list index out of range

In [16]:
q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.\-])*\](\(.*\))')
part = '# [Demon Stores Look for New Hosts](https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020?email={{profile.email}}) |\n\n![Spirit Halloween store](https://dlp31coh2a67q.cloudfront.net/eyJrZXkiOiJ1cGxvYWRzL21lZGl1bS9hc3NldC8yOTkwL3NwaXJpdF9oYWxsb3dlZW5fc3RvcmUuanBnIiwiYnVja2V0Ijoib3Nsby1wcm9kdWN0aW9uIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoxMDAwLCJoZWlnaHQiOm51bGx9fX0=)\n\n_Ben McCanna/Portland Portland Press Herald via Getty Images_\n\n|     |\n| --- |\n| Every year, pop-up Halloween stores rise from the underworld to possess empty storefronts vacated by mortal retailers.\n\nTypically, this non-traditional leasing model means seasonal pop-ups perform well when the rest of retail flounders. But even with 2020 laying out a feast of fresh retail corpses, are spooky stores immune to the pandemic? Let\'s check in with two retail reanimaters:\n\n1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+) than last year, and its CEO said last month the company is "anticipating a Halloween on par with last year."\n2. **Party City** was far less bullish, opening 91% fewer of its Halloween City locations this year. With just 25 stores across the U.S., these days it\'s more like a Halloween Town.\n\n**Zoom out:** Halloween City was showing zombie-like symptoms even before the pandemic. Last year, sales per store declined nearly [21% annually](https://www.retaildive.com/news/party-citys-october-sales-a-disaster-with-halloween-stores-down-21/566955/?utm_source=morning_brew)—a major problem for parent Party City considering Halloween sales account for a fifth of its total U.S. retail revenue.\n\n|     |\n| --- |\n| [![](https://media.sailthru.com/5z8/1k4/4/2/5e865734bf6f7.jpg)](https://www.facebook.com/sharer/sharer.php?u=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86574d1eb82.jpg)](https://twitter.com/intent/tweet?text=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year&url=https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020&via=MorningBrew)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86576e04837.jpg)](https://www.linkedin.com/shareArticle?mini=true&url=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020&title=Halloween+Pop-Up+Shops+Have+Mixed+Success+In+2020&source=www.morningbrew.com&summary=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year)[![](https://media.sailthru.com/5z8/1k4/4/2/5e8657887c532.jpg)](mailto:?subject=Check%20out%20this%20story%20from%20Morning%20Brew%21&body=Halloween%20Pop-Up%20Shops%20Have%20Mixed%20Success%20In%202020%3A%20https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020%0A%0AWant%20more%20great%20content%3F%20Subscribe%20to%20Daily%20Brew%20%E2%80%94%20Delivering%20the%20latest%20business%20news%20from%20Wall%20St.%20to%20Silicon%20Valley%2C%20daily.%0Ahttps%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fr%2F%3Fkid%3D{{profile.vars.referral_code}}%26utm_source%3Demail_share%0A) | | |\n| --- |\n\n| |     |\n| --- |\n| ### FOOD'
matched = q_search.search(part)
matched.group()

'1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+)'

In [ ]:
raw_text = '1. "Overdue VHS tape from 1999 leads to [warrant for embezzlement](https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew)"'

url = re.compile(r'\]\((.*?)\)[,.’"”\\\?:]?[”’\']?(?:\s|$)').search(raw_text).group(1)#.strip("](),. ")
url

# urlsplit(url)

'https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew'

In [163]:
raw_text = '1. "Overdue VHS tape from 1999 leads to [warrant for embezzlement](https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew)"'

start = raw_text.find("](") + 2

depth = 1
for i in range(start, len(raw_text)):
    if raw_text[i] == "(":
        depth += 1
    elif raw_text[i] == ")":
        depth -= 1
        if depth == 0:
            end = i
            break

url = raw_text[start:end]
url

'https://www.nytimes.com/2021/04/25/us/caron-scarborough-davis-vhs-return-warrant.html?utm_source=morning_brew'

In [7]:
with open(f"{data_path}/manual_quiz_additions.json", "r") as f:
    manual_quiz_additions = json.load(f)

print(manual_quiz_additions)

for item in manual_quiz_additions:
    print(item)

new_list = []
print(new_list)
for item in manual_quiz_additions:
    new_list.append(item)

print(new_list)

[{'_id': 'SQTYz1c0U2PEid3hQDOnwo', 'date': '2022-06-30', 'source_url': 'https://www.morningbrew.com/issues/facing-the-music', 'headlines': [{'text': 'Shell is hiring a TikTok manager to reach Gen Z', 'url': 'https://www.bloomberg.com/news/articles/2022-06-29/shell-is-hiring-a-tiktok-channel-manager-to-reach-gen-z?srnd=premium#xj4y7vzkg'}, {'text': 'New Wimbledon documentary has interviews with former pros, journalists, andâ€¦Dick Cheney', 'url': 'https://giphy.com/gifs/wimbledon-dance-tennis-celebrate-Si3yvzzR6HZ1C'}, {'text': 'Analysts accuse Bed Bath & Beyond of turning off AC in stores to save money as sales plummet', 'url': 'https://edition.cnn.com/2022/06/28/investing/bed-bath--beyond-sales-decline/index.html'}, {'text': 'Man loses USB flash drive with data on entire cityâ€™s residents after night out', 'url': 'https://edition.cnn.com/2022/06/24/asia/japan-amagasaki-usb-data-intl-hnk/index.html'}], 'answer': 'We made up the Wimbledon doc one.'}, {'_id': '3wIi2vuFo92PQWFIRJDTrq', '